In [ ]:
import com.hopskipnfall.*
import java.io.File
import kotlin.random.Random

/** Current time as a duration relative to the start of the simulation. */
var now = 0.nanoseconds
val TIME_STEP = 10.microseconds

val frameNumberLogger = TimeBasedDataLogger({ it.toSecondsDouble() })
val frameDriftLogger = TimeBasedDataLogger({ it.toSecondsDouble() })
val objectiveLagLogger = TimeBasedDataLogger({ it.toSecondsDouble() })

val random = Random(
  42L // Use a fixed seed for consistent results between runs.
)

// Some actual ping measurements I took.
val WIFI = LognormalDistribution(mean = 10.424.milliseconds, stdev = 8.193.milliseconds, random)
val WIRED = LognormalDistribution(mean = 6.731.milliseconds, stdev = 1.920.milliseconds, random)

val diagramBuilder = DiagramBuilder()

val clients =
  listOf(
    Client(id = 0, frameDelay = 1, WIFI),
//    Client(id = 1, frameDelay = 1, WIRED),
  )
val server = Server(clients)
for (client in clients) {
  client.server = server
  client.siblings = clients.filter { it.id != client.id }
}

while (now <= 1.minutes) {
  server.run(now, diagramBuilder, frameDriftLogger, objectiveLagLogger)
  for (it in clients) it.run(now, frameNumberLogger, diagramBuilder, objectiveLagLogger, TIME_STEP)

  now += TIME_STEP
}
check(clients.all { it.isHealthy(now) }) {
  "One or more clients is unhealthy! A deadlock likely occurred."
}
server.lagstat(now)



In [ ]:
%use kandy, dataframe
import com.github.nwillc.ksvg.RenderMode
import java.io.FileWriter

if (clients.size < 3) {
  val svg = diagramBuilder.draw()
  FileWriter("diagram.svg").use { svg.render(it, RenderMode.FILE) }
} else {
  println("Not drawing diagram, too many clients.")
}

frameDriftLogger.buildDataFrame().plot {
  line {
    x("Timstamp (seconds)")
    y("Induced gameplay drift")

    color("Client")
  }
}

In [ ]:
%use kandy, dataframe

val df = objectiveLagLogger.buildDataFrame()

df
//df.plot {
//  points {
//    x("Timstamp (seconds)")g
//    y("Objective lag in a single frame (ms)")
//
//    color("Client")
//  }
//
//  layout { title = "Objective lag experienced by clients" }
//}


### Optional: Load in a `GameLag` record:

In [ ]:
import java.io.FileInputStream
import org.emulinker.proto.GameLog

FileInputStream("gamelog_3pall1f.bin").use { fileInputStream ->
  val log = GameLog.parseFrom(fileInputStream.readBytes())
  println("Number of events: ${log.eventsList.size}")
  log.eventsList.take(20).map { it.eventTypeCase }
}